In [1]:
# 从这里开始我们思考如何实现loss函数部分

In [2]:
import torch 

from pytorch3d.io import load_objs_as_meshes
from pytorch3d.structures import Meshes
import drone_renderer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

render = drone_renderer.DroneRenderer(mesh_path= "data/sample/sample.obj",device= device)
mesh = render.mesh

B = 4  # 批量大小
P = torch.rand(B, 3,device=device) *10.0  # 假设的无人机中心点坐标



In [3]:
from pytorch3d.ops import sample_points_from_meshes, knn_points

# 1. 将 Mesh 转换为点云 (Point Cloud) 以供 KNN 使用
# 我们直接使用 render 对象中已经加载好的 mesh
# num_samples 决定了障碍物点云的密度，点越多计算越精确但开销越大
num_samples = 20000 
obstacle_pcd = sample_points_from_meshes(render.mesh, num_samples=num_samples)

print(f"生成的障碍物点云形状: {obstacle_pcd.shape}")  # 预期: (1, 20000, 3)

生成的障碍物点云形状: torch.Size([1, 20000, 3])


In [4]:
p1 = P.unsqueeze(1)  # 形状变为 (B, 1, 3)
p2 = obstacle_pcd  # 形状为 (1, N, 3)，N 是点云中的点数
p2 = p2.expand(B,-1,-1)  # 扩展为 (B, N, 3) 以匹配无人机批量大小
print(f"扩展后的障碍物点云形状: {p2.shape}")  # 预期: (B, 20000, 3)

dists = knn_points(p1, p2, K=1)
print(f"计算得到的最近距离形状: {dists.dists.shape}")  # 预期: (B, 1, 1)
print("距离结果",dists)
print("纯距离",dists.dists)  # 打印距离值以检查
print("形状",dists.dists.shape)
dists.dists.squeeze(-1)
print("去掉最后一个维度后的形状",dists.dists.squeeze(-1).shape)



扩展后的障碍物点云形状: torch.Size([4, 20000, 3])
计算得到的最近距离形状: torch.Size([4, 1, 1])
距离结果 KNN(dists=tensor([[[11.9785]],

        [[31.5639]],

        [[ 5.6733]],

        [[54.1752]]], device='cuda:0'), idx=tensor([[[16948]],

        [[ 1098]],

        [[ 4376]],

        [[ 4890]]], device='cuda:0'), knn=None)
纯距离 tensor([[[11.9785]],

        [[31.5639]],

        [[ 5.6733]],

        [[54.1752]]], device='cuda:0')
形状 torch.Size([4, 1, 1])
去掉最后一个维度后的形状 torch.Size([4, 1])


In [ ]:

def calc_min_distance(drone_pos, obstacle_pcd):
    """
    计算无人机中心点与障碍物点云之间的最短距离 (Single Step)
    arges:
        drone_pos: (B, 3) 无人机中心点坐标
        obstacle_pcd: (1 or B, N, 3) 障碍物点云
    returns:
        dists: (B,) 每个无人机到障碍物的最短距离
    """
    p1 = drone_pos.unsqueeze(1) 
    B = p1.shape[0]
    
    if obstacle_pcd.shape[0] != B:
        obstacle_pcd_expanded = obstacle_pcd.expand(B, -1, -1)
    else:
        obstacle_pcd_expanded = obstacle_pcd
        
    result = knn_points(p1, obstacle_pcd_expanded, K=1)
    sq_dists = result.dists.squeeze(-1) # (B, 1) -> (B,)
    dists = torch.sqrt(sq_dists + 1e-6).squeeze(-1) 
    return dists

min_distances = calc_min_distance(P, obstacle_pcd)
print("部分无人机到障碍物的最短距离 (Single Step):", min_distances)

部分无人机到障碍物的最短距离 (Single Step): tensor([3.4610, 5.6182, 2.3819, 7.3604], device='cuda:0')
